# Notebook 43: Tenancy

This notebook demonstrates multi-tenant isolation patterns using `multigen.tenancy`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `TenantRegistry` — create, list, and delete tenants
- `TenantScope` — async context manager for scoping the current tenant
- `UsageTracker` — quotas, recording usage, and `QuotaExceededError`
- `TenantAwareRegistry` — per-tenant agent registration with isolation
- `TenantManager` facade — full onboard/offboard lifecycle

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.tenancy`

We define all classes locally so the notebook is fully self-contained.

In [ ]:
import asyncio
import uuid
from contextlib import asynccontextmanager
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

# ---------------------------------------------------------------------------
# Exceptions
# ---------------------------------------------------------------------------

class QuotaExceededError(Exception):
    """Raised when a tenant exceeds their token quota."""

# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------

@dataclass
class Tenant:
    id: str
    name: str
    plan: str  # e.g. 'free', 'pro', 'enterprise'

# ---------------------------------------------------------------------------
# 1. TenantRegistry
# ---------------------------------------------------------------------------

class TenantRegistry:
    def __init__(self):
        self._tenants: Dict[str, Tenant] = {}

    def create_tenant(self, name: str, plan: str = 'free') -> Tenant:
        tenant_id = str(uuid.uuid4())[:8]
        t = Tenant(id=tenant_id, name=name, plan=plan)
        self._tenants[tenant_id] = t
        return t

    def list_tenants(self) -> List[Tenant]:
        return list(self._tenants.values())

    def delete_tenant(self, tenant_id: str) -> bool:
        if tenant_id in self._tenants:
            del self._tenants[tenant_id]
            return True
        return False

    def get(self, tenant_id: str) -> Optional[Tenant]:
        return self._tenants.get(tenant_id)

print('TenantRegistry defined.')


## Section 1 — TenantRegistry

In [ ]:
registry = TenantRegistry()

acme = registry.create_tenant('Acme Corp', plan='pro')
beta = registry.create_tenant('Beta Labs', plan='free')
gamma = registry.create_tenant('Gamma AI', plan='enterprise')

print('Created tenants:')
for t in registry.list_tenants():
    print(f'  id={t.id}  name={t.name!r:20s}  plan={t.plan}')

deleted = registry.delete_tenant(beta.id)
print(f'\nDeleted beta tenant (id={beta.id}): {deleted}')
print('Remaining tenants:', [t.name for t in registry.list_tenants()])


## Section 2 — TenantScope (async context manager)

In [ ]:
# Thread-local-like storage for async context
_current_tenant: Optional[Tenant] = None

def current_tenant() -> Optional[Tenant]:
    return _current_tenant

class TenantScope:
    """Async context manager that sets the active tenant for the duration of a block."""
    def __init__(self, tenant: Tenant):
        self._tenant = tenant
        self._previous: Optional[Tenant] = None

    async def __aenter__(self):
        global _current_tenant
        self._previous = _current_tenant
        _current_tenant = self._tenant
        return self._tenant

    async def __aexit__(self, *_):
        global _current_tenant
        _current_tenant = self._previous


async def demo_scope():
    print(f'Before scope: current_tenant = {current_tenant()}')

    async with TenantScope(acme):
        print(f'Inside acme scope: current_tenant = {current_tenant().name}')

        async with TenantScope(gamma):
            print(f'  Inside gamma scope: current_tenant = {current_tenant().name}')

        print(f'Back to acme scope: current_tenant = {current_tenant().name}')

    print(f'After scope: current_tenant = {current_tenant()}')

await demo_scope()


## Section 3 — UsageTracker

In [ ]:
@dataclass
class UsageSummary:
    tenant_id: str
    quota: int
    used: int

    @property
    def tokens_remaining(self) -> int:
        return max(0, self.quota - self.used)

    @property
    def over_quota(self) -> bool:
        return self.used > self.quota


class UsageTracker:
    def __init__(self):
        self._quotas: Dict[str, int] = {}
        self._usage: Dict[str, int] = {}

    def set_quota(self, tenant_id: str, tokens: int):
        self._quotas[tenant_id] = tokens
        self._usage.setdefault(tenant_id, 0)

    def record_usage(self, tenant_id: str, tokens: int):
        self._usage[tenant_id] = self._usage.get(tenant_id, 0) + tokens
        quota = self._quotas.get(tenant_id, 0)
        if self._usage[tenant_id] > quota:
            raise QuotaExceededError(
                f'Tenant {tenant_id} used {self._usage[tenant_id]} tokens '
                f'(quota={quota})'
            )

    def summary(self, tenant_id: str) -> UsageSummary:
        return UsageSummary(
            tenant_id=tenant_id,
            quota=self._quotas.get(tenant_id, 0),
            used=self._usage.get(tenant_id, 0),
        )


tracker = UsageTracker()
tracker.set_quota(acme.id, quota=10_000)
tracker.set_quota(gamma.id, quota=500_000)

tracker.record_usage(acme.id, 3_000)
tracker.record_usage(acme.id, 4_000)

s = tracker.summary(acme.id)
print(f'Acme usage summary:')
print(f'  quota          = {s.quota:,}')
print(f'  used           = {s.used:,}')
print(f'  tokens_remaining = {s.tokens_remaining:,}')
print(f'  over_quota     = {s.over_quota}')

# Now exceed the quota
print('\nAttempting to exceed quota …')
try:
    tracker.record_usage(acme.id, 5_000)   # total = 12 000 > 10 000
except QuotaExceededError as e:
    print(f'QuotaExceededError caught: {e}')

s2 = tracker.summary(acme.id)
print(f'over_quota after breach: {s2.over_quota}')


## Section 4 — TenantAwareRegistry

In [ ]:
class TenantAwareRegistry:
    """Registers agents per tenant; each tenant sees only its own agents."""

    def __init__(self):
        self._agents: Dict[str, Dict[str, Any]] = {}

    def register_agent(self, tenant_id: str, agent_name: str, agent_fn):
        self._agents.setdefault(tenant_id, {})[agent_name] = agent_fn

    def list_agents(self, tenant_id: str) -> List[str]:
        return list(self._agents.get(tenant_id, {}).keys())

    def resolve(self, tenant_id: str, agent_name: str):
        bucket = self._agents.get(tenant_id, {})
        if agent_name not in bucket:
            raise KeyError(f'Agent {agent_name!r} not found for tenant {tenant_id}')
        return bucket[agent_name]


def mock_summariser(text: str) -> str:
    return f'[SUMMARY of {len(text)} chars]'

def mock_classifier(text: str) -> str:
    return '[CLASS: positive]'

def mock_translator(text: str) -> str:
    return '[ES] ' + text


ta_registry = TenantAwareRegistry()

# Acme gets summariser + classifier
ta_registry.register_agent(acme.id, 'summariser', mock_summariser)
ta_registry.register_agent(acme.id, 'classifier', mock_classifier)

# Gamma gets summariser + translator
ta_registry.register_agent(gamma.id, 'summariser', mock_summariser)
ta_registry.register_agent(gamma.id, 'translator', mock_translator)

print(f'Acme agents  : {ta_registry.list_agents(acme.id)}')
print(f'Gamma agents : {ta_registry.list_agents(gamma.id)}')

# Gamma cannot see classifier (isolation)
print('\nGamma trying to resolve "classifier" (should fail):')
try:
    ta_registry.resolve(gamma.id, 'classifier')
except KeyError as e:
    print(f'  KeyError: {e}')

# Acme uses summariser
fn = ta_registry.resolve(acme.id, 'summariser')
print(f'\nAcme summariser result: {fn("The quick brown fox jumps over the lazy dog.")}')


## Section 5 — TenantManager Facade

In [ ]:
class TenantManager:
    """
    Facade that combines TenantRegistry, TenantScope, UsageTracker,
    and TenantAwareRegistry into a single interface.
    """

    def __init__(self):
        self.registry = TenantRegistry()
        self.usage = UsageTracker()
        self.agents = TenantAwareRegistry()

    def create_tenant(self, name: str, plan: str = 'free', quota: int = 50_000) -> Tenant:
        t = self.registry.create_tenant(name, plan)
        self.usage.set_quota(t.id, quota)
        print(f'[TenantManager] Onboarded tenant {t.name!r} (id={t.id}, plan={t.plan}, quota={quota:,})')
        return t

    def scope(self, tenant: Tenant) -> TenantScope:
        return TenantScope(tenant)

    def record_usage(self, tenant: Tenant, tokens: int):
        self.usage.record_usage(tenant.id, tokens)

    def usage_report(self, tenant: Tenant) -> UsageSummary:
        return self.usage.summary(tenant.id)

    def delete_tenant(self, tenant: Tenant):
        self.registry.delete_tenant(tenant.id)
        print(f'[TenantManager] Offboarded tenant {tenant.name!r} (id={tenant.id})')


async def full_lifecycle():
    mgr = TenantManager()

    # Onboard
    client_a = mgr.create_tenant('ClientA', plan='pro', quota=20_000)
    client_b = mgr.create_tenant('ClientB', plan='free', quota=5_000)

    # Register agents
    mgr.agents.register_agent(client_a.id, 'summariser', mock_summariser)
    mgr.agents.register_agent(client_b.id, 'classifier', mock_classifier)

    # Scoped execution for client_a
    async with mgr.scope(client_a):
        print(f'\nExecuting as: {current_tenant().name}')
        mgr.record_usage(client_a, 8_000)
        report = mgr.usage_report(client_a)
        print(f'  used={report.used:,}  remaining={report.tokens_remaining:,}  over_quota={report.over_quota}')

    # Scoped execution for client_b
    async with mgr.scope(client_b):
        print(f'\nExecuting as: {current_tenant().name}')
        mgr.record_usage(client_b, 3_000)
        report = mgr.usage_report(client_b)
        print(f'  used={report.used:,}  remaining={report.tokens_remaining:,}  over_quota={report.over_quota}')

    # Offboard client_b
    print()
    mgr.delete_tenant(client_b)
    print('Remaining tenants:', [t.name for t in mgr.registry.list_tenants()])

await full_lifecycle()
